# Humana (HUM) — Indicative Bond Pricing Walkthrough

This notebook manually walks through the methodology for generating indicative new-issue yields for Humana Inc. (HUM), a managed-care issuer rated BBB by S&P. We start by pulling the current U.S. Treasury curve from FRED, define a peer set of comparable investment-grade healthcare names, fit a Nelson-Siegel spread curve to observed peer bond spreads, and layer on a credit adjustment and new-issue concession (NIC) to arrive at indicative all-in yields across the curve. The core formula is:

$$\text{Indicative Yield} = \text{Treasury Rate} + \text{Peer Spread} + \text{Credit Adjustment} + \text{NIC}$$

The final output is compared to Humana's published Q4 2025 pricing deck to validate the approach.

---
## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import requests
from datetime import date

# ---------- Configuration ----------
# Get a free key at: https://fred.stlouisfed.org/docs/api/api_key.html
FRED_API_KEY: str = "YOUR_FRED_API_KEY_HERE"

FRED_BASE_URL = "https://api.stlouisfed.org/fred/series/observations"

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print(f"Analysis date: {date.today()}")

---
## Step 1 — Pull the Treasury Curve

We pull the most recent daily constant-maturity Treasury yields from FRED for the standard benchmark tenors. These serve as the risk-free base rates that we add credit spreads on top of.

In [ ]:
TENOR_MAP: dict[str, int] = {
    "DGS2": 2,
    "DGS5": 5,
    "DGS7": 7,
    "DGS10": 10,
    "DGS20": 20,
    "DGS30": 30,
}

treasury_data: list[dict] = []
for series_id, tenor in TENOR_MAP.items():
    resp = requests.get(FRED_BASE_URL, params={
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "sort_order": "desc",
        "limit": 10,
    })
    resp.raise_for_status()
    obs = [o for o in resp.json()["observations"] if o["value"] != "."]
    latest = obs[0]
    treasury_data.append({
        "tenor": tenor,
        "series": series_id,
        "yield_pct": round(float(latest["value"]), 3),
        "as_of": latest["date"],
    })

treasury_df = pd.DataFrame(treasury_data)
treasury_df

In [ ]:
fig, ax = plt.subplots()
ax.plot(treasury_df["tenor"], treasury_df["yield_pct"], "o-", color="#1f77b4", linewidth=2)
ax.set_xlabel("Tenor (years)")
ax.set_ylabel("Yield (%)")
ax.set_title("U.S. Treasury Constant-Maturity Curve")
ax.set_xticks(treasury_df["tenor"])
for _, row in treasury_df.iterrows():
    ax.annotate(f"{row['yield_pct']:.2f}%", (row["tenor"], row["yield_pct"]),
                textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

---
## Step 2 — Define the Peer Set

We define a set of comparable managed-care issuers with outstanding bonds across the curve. In production, this data would come from Bloomberg SRCH or FINRA TRACE; here we hard-code representative values for the prototype.

**Peers:** UnitedHealth (UNH), Elevance Health (ELV), Cigna (CI), Centene (CNC), Molina Healthcare (MOH).

In [ ]:
peer_bonds = pd.DataFrame([
    # UnitedHealth Group — A+
    {"issuer": "UnitedHealth Group", "ticker": "UNH", "coupon": 3.50, "maturity_date": "2027-06-15", "years_to_maturity": 2.1, "current_spread_bps": 52, "rating": "A+"},
    {"issuer": "UnitedHealth Group", "ticker": "UNH", "coupon": 4.25, "maturity_date": "2032-03-15", "years_to_maturity": 6.9, "current_spread_bps": 78, "rating": "A+"},
    {"issuer": "UnitedHealth Group", "ticker": "UNH", "coupon": 4.75, "maturity_date": "2045-07-15", "years_to_maturity": 20.2, "current_spread_bps": 105, "rating": "A+"},

    # Elevance Health — A-
    {"issuer": "Elevance Health",    "ticker": "ELV", "coupon": 3.35, "maturity_date": "2027-12-01", "years_to_maturity": 2.6, "current_spread_bps": 58, "rating": "A-"},
    {"issuer": "Elevance Health",    "ticker": "ELV", "coupon": 4.10, "maturity_date": "2032-05-15", "years_to_maturity": 7.0, "current_spread_bps": 85, "rating": "A-"},
    {"issuer": "Elevance Health",    "ticker": "ELV", "coupon": 5.50, "maturity_date": "2054-01-15", "years_to_maturity": 28.7, "current_spread_bps": 128, "rating": "A-"},

    # Cigna Group — BBB+
    {"issuer": "Cigna Group",        "ticker": "CI",  "coupon": 3.75, "maturity_date": "2027-07-15", "years_to_maturity": 2.2, "current_spread_bps": 65, "rating": "BBB+"},
    {"issuer": "Cigna Group",        "ticker": "CI",  "coupon": 4.50, "maturity_date": "2034-02-25", "years_to_maturity": 8.8, "current_spread_bps": 102, "rating": "BBB+"},
    {"issuer": "Cigna Group",        "ticker": "CI",  "coupon": 5.40, "maturity_date": "2054-03-15", "years_to_maturity": 28.9, "current_spread_bps": 142, "rating": "BBB+"},

    # Centene — BBB-
    {"issuer": "Centene",            "ticker": "CNC", "coupon": 4.25, "maturity_date": "2027-12-15", "years_to_maturity": 2.5, "current_spread_bps": 80, "rating": "BBB-"},
    {"issuer": "Centene",            "ticker": "CNC", "coupon": 4.625,"maturity_date": "2029-12-15", "years_to_maturity": 4.6, "current_spread_bps": 110, "rating": "BBB-"},
    {"issuer": "Centene",            "ticker": "CNC", "coupon": 3.00, "maturity_date": "2030-10-15", "years_to_maturity": 5.4, "current_spread_bps": 118, "rating": "BBB-"},

    # Molina Healthcare — BBB-
    {"issuer": "Molina Healthcare",  "ticker": "MOH", "coupon": 3.875,"maturity_date": "2028-11-15", "years_to_maturity": 3.5, "current_spread_bps": 95, "rating": "BBB-"},
    {"issuer": "Molina Healthcare",  "ticker": "MOH", "coupon": 4.375,"maturity_date": "2032-06-15", "years_to_maturity": 7.1, "current_spread_bps": 130, "rating": "BBB-"},
])

peer_bonds["maturity_date"] = pd.to_datetime(peer_bonds["maturity_date"])
print(f"{len(peer_bonds)} peer bonds across {peer_bonds['ticker'].nunique()} issuers")
peer_bonds.sort_values("years_to_maturity")

> **Note:** In production, peer bond data (spreads, ratings, outstanding amounts) would be sourced from Bloomberg `SRCH` or FINRA TRACE. The values above are representative placeholders calibrated to recent market levels.

---
## Step 3 — Visualize Peer Spreads

Before fitting, we plot the raw peer spread data to check for outliers and confirm the expected upward-sloping term structure.

In [ ]:
fig, ax = plt.subplots()

colors = {"UNH": "#1f77b4", "ELV": "#ff7f0e", "CI": "#2ca02c", "CNC": "#d62728", "MOH": "#9467bd"}

for ticker, group in peer_bonds.groupby("ticker"):
    ax.scatter(group["years_to_maturity"], group["current_spread_bps"],
               color=colors[ticker], label=f"{ticker} ({group['rating'].iloc[0]})",
               s=80, edgecolors="white", zorder=3)

ax.set_xlabel("Years to Maturity")
ax.set_ylabel("Spread (bps)")
ax.set_title("Peer Bond Spreads vs. Maturity")
ax.legend(title="Issuer (Rating)", fontsize=9)
plt.tight_layout()
plt.show()

---
## Step 4 — Fit a Nelson-Siegel Spread Curve

We fit the Nelson-Siegel functional form to the peer spread data. This gives us a smooth parametric curve that captures the level, slope, and curvature of the spread term structure:

$$y(\tau) = \beta_0 + \beta_1 \cdot \frac{1 - e^{-\tau/\lambda}}{\tau/\lambda} + \beta_2 \cdot \left[\frac{1 - e^{-\tau/\lambda}}{\tau/\lambda} - e^{-\tau/\lambda}\right]$$

We fix $\lambda = 1.37$ (a standard starting value for medium-term IG curves) and estimate $\beta_0$, $\beta_1$, $\beta_2$ via least-squares.

In [ ]:
LAMBDA: float = 1.37


def nelson_siegel(tau: np.ndarray, beta0: float, beta1: float, beta2: float) -> np.ndarray:
    """Nelson-Siegel spread model with fixed lambda."""
    tau_ratio = tau / LAMBDA
    factor1 = (1 - np.exp(-tau_ratio)) / tau_ratio
    factor2 = factor1 - np.exp(-tau_ratio)
    return beta0 + beta1 * factor1 + beta2 * factor2


tau_obs = peer_bonds["years_to_maturity"].values
spread_obs = peer_bonds["current_spread_bps"].values

popt, pcov = curve_fit(nelson_siegel, tau_obs, spread_obs, p0=[100, -50, 30])
beta0, beta1, beta2 = popt

print(f"Fitted parameters (lambda = {LAMBDA}):")
print(f"  beta_0 (long-run level) = {beta0:.2f} bps")
print(f"  beta_1 (short-term)     = {beta1:.2f} bps")
print(f"  beta_2 (hump)           = {beta2:.2f} bps")

In [ ]:
tau_smooth = np.linspace(1, 32, 200)
spread_fitted = nelson_siegel(tau_smooth, *popt)

fig, ax = plt.subplots()
for ticker, group in peer_bonds.groupby("ticker"):
    ax.scatter(group["years_to_maturity"], group["current_spread_bps"],
               color=colors[ticker], label=f"{ticker} ({group['rating'].iloc[0]})",
               s=80, edgecolors="white", zorder=3)
ax.plot(tau_smooth, spread_fitted, "k--", linewidth=1.5, label="Nelson-Siegel fit")
ax.set_xlabel("Years to Maturity")
ax.set_ylabel("Spread (bps)")
ax.set_title("Nelson-Siegel Fit to Peer Spread Data")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Step 5 — Predict Humana Indicative Yields

We use the fitted Nelson-Siegel curve to read off predicted peer spreads at each benchmark tenor, then layer on:

- **Credit adjustment:** +5 bps. Humana is rated BBB, slightly weaker than the peer-set median (BBB+/A-), which warrants a small upward adjustment.
- **New-issue concession (NIC):** +5 bps. Standard concession to compensate investors for taking down new paper vs. buying seasoned bonds in the secondary market.

Final spread = Peer spread + Credit adj. + NIC, and:

$$\text{Indicative Yield} = \text{Treasury Yield} + \text{Final Spread (in %)}$$

In [ ]:
CREDIT_ADJ_BPS: int = 5
NIC_BPS: int = 5

tenors = treasury_df["tenor"].values
treasury_yields = treasury_df["yield_pct"].values

peer_spreads = nelson_siegel(tenors.astype(float), *popt)

output = pd.DataFrame({
    "tenor": tenors,
    "treasury_yield": treasury_yields,
    "peer_spread_bps": np.round(peer_spreads, 1),
    "credit_adj_bps": CREDIT_ADJ_BPS,
    "nic_bps": NIC_BPS,
})

output["final_spread_bps"] = output["peer_spread_bps"] + output["credit_adj_bps"] + output["nic_bps"]
output["indicative_yield"] = np.round(output["treasury_yield"] + output["final_spread_bps"] / 100, 3)

output

---
## Step 6 — Compare to Actual Humana Deck

We compare our model-implied spreads to the spreads from Humana's published Q4 2025 pricing deck. This is the key validation: if our model is directionally correct and within a reasonable band (say +/- 15 bps), the methodology holds up for a first pass.

In [ ]:
deck_spreads_bps: dict[int, int] = {
    2: 50,
    5: 89,
    7: 104,
    10: 135,
    20: 143,
    30: 155,
}

comparison = output[["tenor", "final_spread_bps", "indicative_yield"]].copy()
comparison["deck_spread_bps"] = comparison["tenor"].map(deck_spreads_bps)
comparison["diff_bps"] = np.round(comparison["final_spread_bps"] - comparison["deck_spread_bps"], 1)

comparison.rename(columns={
    "final_spread_bps": "model_spread_bps",
}, inplace=True)

print("Model vs. Humana Q4 2025 Deck (spreads in bps):")
comparison

In [ ]:
fig, ax = plt.subplots()
ax.plot(comparison["tenor"], comparison["model_spread_bps"], "o-",
        color="#1f77b4", linewidth=2, label="Model predicted")
ax.plot(comparison["tenor"], comparison["deck_spread_bps"], "s--",
        color="#d62728", linewidth=2, label="Humana Q4 2025 deck")
ax.fill_between(comparison["tenor"],
                comparison["deck_spread_bps"] - 15,
                comparison["deck_spread_bps"] + 15,
                color="#d62728", alpha=0.08, label="+/- 15 bps band")
ax.set_xlabel("Tenor (years)")
ax.set_ylabel("Spread (bps)")
ax.set_title("Model vs. Actual — Humana Indicative Spreads")
ax.set_xticks(comparison["tenor"])
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Next Steps & Open Questions

- **What this shows:** We can replicate the shape and approximate level of Humana's published spread curve using a Nelson-Siegel fit to peer data plus a simple credit/NIC overlay. The methodology is transparent and auditable.
- **What's missing vs. production:** Real-time Bloomberg/TRACE data feeds, automated peer screening by sector + rating + maturity buckets, historical back-testing of fitted vs. realized spreads, broader issuer coverage beyond managed care, and sensitivity analysis on the credit adjustment.
- **Open methodology questions for discussion:**
  - Should $\lambda$ be estimated or kept fixed? Estimating it adds a free parameter but may overfit with only ~14 peer bonds.
  - Is a flat +5 bps credit adjustment appropriate, or should we use a rating-transition-matrix-based approach?
  - How should we weight peer bonds — equal weight, inverse variance, or by issue size / liquidity?